In [21]:
import pandas as pd
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

load_dotenv()  # loads variables from .env


# Configuration
NEO4J_URL = os.getenv("NEO4J_URL")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
api_key = os.getenv("OPENAI_API_KEY")
groq_api_key  = os.getenv("GROQ_API_KEY")

graph = Neo4jGraph(url=NEO4J_URL, username=NEO4J_USER, password=NEO4J_PASSWORD, database="78ce8520",
    refresh_schema=True )

In [22]:
print(graph.schema)

Node properties:
Country {name: STRING, region: STRING}
YearMonth {yearMonth: INTEGER}
Relationship properties:
IMPORTED_FROM {barrels: FLOAT, value_usd: FLOAT, price_mt: FLOAT, year_month: INTEGER}
The relationships:
(:Country)-[:IMPORTS_IN]->(:YearMonth)
(:Country)-[:IMPORTED_FROM]->(:Country)


In [23]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_groq import ChatGroq


graph.refresh_schema()

examples = [
    {
        "question": "Show me total volume and average price for Russia to India trade.",
        "query": """
        MATCH (t:YearMonth)
        WITH max(t.yearMonth) / 100 AS latest_year
        // We aggregate all months in the latest_year to avoid single-month bias
        MATCH (importer:Country {{name: 'India'}})-[:IMPORTS_IN]->(tm:YearMonth)
        WHERE tm.yearMonth / 100 = latest_year
        MATCH (importer)-[f:IMPORTED_FROM {{year_month: tm.yearMonth}}]->(exporter:Country {{name: 'Russian Federation'}})
        RETURN ROUND(sum(f.barrels)/1000000,2) AS total_volume_barrels_in_millions, 
            ROUND(avg(f.price_mt),2) AS average_price_in_dollars
        """
    },
    {
        "question": "Which countries were active importers in September 2023?",
        "query": """
        MATCH (importer:Country)-[:IMPORTS_IN]->(t:YearMonth {{yearMonth: 202309}})
        MATCH (importer)-[f:IMPORTED_FROM {{year_month: 202309}}]->(exporter:Country)
        RETURN importer.name AS country, 
           ROUND(sum(f.barrels)/1000000,2) AS total_barrels_imported_in_millions,
           count(exporter) AS number_of_suppliers
        ORDER BY total_barrels_imported_in_millions DESC 
        LIMIT 5"""
    },
    {
        "question": "Which country exported the most barrels to the USA in Jan 2024?",
        "query": """
        MATCH (importer:Country {{name: 'USA'}})-[:IMPORTS_IN]->(t:YearMonth {{yearMonth: 202401}})
        MATCH (importer)-[f:IMPORTED_FROM {{year_month: 202401}}]->(exporter:Country)
        RETURN exporter.name AS exporter, ROUND(f.barrels/1000000,2) AS  total_barrels_imported_in_millions
        ORDER BY total_barrels_imported_in_millions DESC LIMIT 1
        """
    },
    {
        "question": "How much oil did India import from the Russians in 202305?",
        "query": """
        MATCH (importer:Country {{name: 'India'}})-[:IMPORTS_IN]->(t:YearMonth {{yearMonth: 202305}})
        MATCH (importer)-[f:IMPORTED_FROM {{year_month: 202305}}]->(exporter:Country {{name: 'Russian Federation'}})
        RETURN ROUND(f.barrels/1000000,2) AS total_barrels_imported_in_millions
        """
    },
{
        "question": "Which countries have the highest risk or potential loss based on reliance on Russia?",
        "query":        """
        MATCH (t:YearMonth)
        WITH max(t.yearMonth) / 100 AS latest_year
        MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
        WHERE tm.yearMonth / 100 = latest_year
        MATCH (importer)-[f:IMPORTED_FROM {{year_month: tm.yearMonth}}]->(exporter:Country)
        WITH importer,
             sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
             sum(f.barrels) AS total_vol
        // We filter for total_vol > 1M to remove tiny partners like Georgia/Armenia
        WHERE total_vol > 1000000
        RETURN importer.name AS country, 
               ROUND(russian_vol / 1000000, 2) AS barrels_from_russia_in_millions,
               ROUND((russian_vol / total_vol) * 100, 2) AS reliance_percent
        ORDER BY barrels_from_russia_in_millions DESC 
        LIMIT 5
        """
    }
]

prompt_template = PromptTemplate(
    input_variables=["question", "query"],
    template="User Question: {question}\nCypher Query: {query}"
)

# 2. Build the Few-Shot Prompt
cypher_prompt = FewShotPromptTemplate(
    examples=examples,

    example_prompt=prompt_template,

    prefix="""
    You are a Neo4j expert.
    Your task is to convert English questions into high-performance Cypher queries based on the provided schema.
    
    CRITICAL RULES:
    1. RELATIONSHIP DIRECTION: All trade relationships follow the pattern: `(Importer:Country)-[:IMPORTED_FROM]->(Exporter:Country)`
    2. NAMES: Map common country names to database values (e.g., 'Russia' or 'Russians' must be 'Russian Federation').
    3. PERFORMANCE: Always use the `(Importer)-[:IMPORTS_IN]->(t:YearMonth)` relationship to filter by time before performing aggregations. 
        This is more efficient than filtering only by relationship properties.
    4. Use `f.barrels` for volume, `f.price_mt` for price, and `f.year_month` for the period identifier.
    5. TEMPORAL DEFAULT: If the user DOES NOT specify a year/month, you MUST aggregate data for the ENTIRE LATEST CALENDAR YEAR.
    - USE: "MATCH (t:YearMonth) WITH max(t.yearMonth) / 100 AS latest_year"
    - DO NOT USE: "max(t.yearMonth)" alone, as this only returns one month.
    - Ensure you sum the volumes across all months where "tm.yearMonth / 100 = latest_year".
    6. UNITS: All volume outputs MUST be converted to **Millions of Barrels**. Divide the raw barrels by 1000000 (e.g., `f.barrels / 1000000`).
            Apply the `ROUND(value, 2)` function to all numerical results, including total volumes and average prices, to ensure clean output.
    7. OUTPUT: Use descriptive aliases for returned values (e.g., `total_barrels_in_millions`, `average_price_usd`).
            Return ONLY the Cypher query. No markdown code blocks (```), no explanations.
    
    Always use LIMIT 5 unless asked otherwise.

    Schema: 
    {schema}
    """, 

    suffix="Question: {question}\nCypher Query: ",

    input_variables=["question", "schema"]
)

CYPHER_QA_TEMPLATE = """
Using the Neo4j results below, answer the user's question.

RULES:
1. If the data covers a full year (e.g. results are large or based on latest_year), start with "Based on total trade for [Year]...".
2. Always append 'million barrels' to volume figures.
3. Format YYYYMM as 'Month YYYY' (e.g. 202401 as Jan 2024).
4. You MUST maintain the exact order of the countries provided in the context. Do not re-rank them.

FINAL FORMATTING CHECK:
- YOU MUST START your response with: "Based on total trade for [Year]/[Month YYYY]..."
- YOU ARE FORBIDDEN from using any bolding (**) or markdown headers. 

User Question: {question}
Context: {context}

Final Answer:"""

qa_prompt_template = PromptTemplate(
    input_variables=["question", "context"], 
    template=CYPHER_QA_TEMPLATE
)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

# 3. Use a powerful model. Maybe it can help in latency
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt_template,
    verbose=True,
    top_k=10, 
    enhanced_schema=True, 
    return_intermediate_steps=True,
    allow_dangerous_requests = True 
)

# Implementation of a "Safety Net" Wrapper
from langchain_community.callbacks.manager import get_openai_callback

def ask_sentinel(question):
    with get_openai_callback() as cb:
        try:
            response = chain.invoke({"query": question})
            
            # Print the token stats
            print(f"--- Token Usage for this Request ---")

            print(f"Prompt Tokens: {cb.prompt_tokens}")
            print(f"Completion Tokens: {cb.completion_tokens}")
            print(f"Total Tokens: {cb.total_tokens}")
            print(f"Total Cost (if applicable): ${cb.total_cost}")

            usage_pct = (cb.total_tokens / 128000) * 100
            print(f"Context Usage: {usage_pct:.2f}%")
            print(f"------------------------------------")

            if not response.get('result') or response['result'] == "I don't know the answer.":
                return "No data found for those parameters."
            
            return response['result']

        except Exception as e:
            print(f"DEBUG: Internal Error -> {str(e)}")
            return "I encountered a technical difficulty. Please try rephrasing."


In [30]:
res = ask_sentinel("Which country provided the most barrels to the USA in 202401?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country {name: 'USA'})-[:IMPORTS_IN]->(t:YearMonth {yearMonth: 202401})
MATCH (importer)-[f:IMPORTED_FROM {year_month: t.yearMonth}]->(exporter:Country)
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels) / 1000000, 2) AS total_barrels_in_millions
ORDER BY total_barrels_in_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Canada', 'total_barrels_in_millions': 126.42}, {'exporter': 'Mexico', 'total_barrels_in_millions': 16.6}, {'exporter': 'Saudi Arabia', 'total_barrels_in_millions': 9.27}, {'exporter': 'Colombia', 'total_barrels_in_millions': 7.71}, {'exporter': 'Brazil', 'total_barrels_in_millions': 7.71}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1746
Completion Tokens: 668
Total Tokens: 2414
Total Cost (if applicable): $0.0
Context Usage: 1.89%
------------------------------------
Based on total trade for Jan 2024, Canada provided the most barrels to the USA, delive

In [31]:
res = ask_sentinel("How much barrels India imported in 202305?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country {name: 'India'})-[:IMPORTS_IN]->(t:YearMonth {yearMonth: 202305})
MATCH (importer)-[f:IMPORTED_FROM {year_month: 202305}]->(exporter:Country)
RETURN ROUND(sum(f.barrels)/1000000, 2) AS total_barrels_imported_in_millions
Full Context:
[{'total_barrels_imported_in_millions': 161.24}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1647
Completion Tokens: 566
Total Tokens: 2213
Total Cost (if applicable): $0.0
Context Usage: 1.73%
------------------------------------
Based on total trade for May 2023, India imported 161.24 million barrels.


In [29]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their number of barrels")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {name: 'Russian Federation'})
WITH importer.name AS country, sum(f.barrels) AS total_barrels
RETURN country,
       ROUND(total_barrels/1000000, 2) AS imported_from_russia_millions,
       ROUND(total_barrels * 0.2 / 1000000, 2) AS potential_loss_millions
ORDER BY potential_loss_millions DESC
LIMIT 5
Full Context:
[{'country': 'China', 'imported_from_russia_millions': 789.66, 'potential_loss_millions': 157.93}, {'country': 'India', 'imported_from_russia_millions': 669.82, 'potential_loss_millions': 133.96}, {'country': 'Hungary', 'imported_from_russia_millions': 35.07, 'potential_loss_millions': 7.01}, {'country': 'Slovakia', 'imported_from_russia_millions': 30.92, 'potential_loss_millions'

In [27]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their historical reliance?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_barrels,
     sum(f.barrels) AS total_barrels
WHERE total_barrels > 0
WITH importer,
     russian_barrels,
     total_barrels,
     (russian_barrels / total_barrels) * 100 AS reliance_percent,
     russian_barrels * 0.2 AS projected_loss_barrels
RETURN importer.name AS country,
       ROUND(russian_barrels/1000000,2) AS barrels_from_russia_in_millions,
       ROUND(total_barrels/1000000,2) AS total_imported_barrels_in_millions,
       ROUND(reliance_percent,2) AS reliance_percent,
       ROUND(projected_loss_barrels/1000000,2) AS projected_loss_millions
ORDER BY reliance_percent DES

In [28]:
res = ask_sentinel("Which country is most reliable on Russia for their imports?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
RETURN importer.name AS country,
       ROUND(russian_vol/1000000,2) AS barrels_from_russia_in_millions,
       ROUND((russian_vol/total_vol)*100,2) AS reliance_percent
ORDER BY reliance_percent DESC, barrels_from_russia_in_millions DESC
LIMIT 5
Full Context:
[{'country': 'Georgia', 'barrels_from_russia_in_millions': 0.08, 'reliance_percent': 100.0}, {'country': 'Armenia', 'barrels_from_russia_in_millions': 0.0, 'reliance_percent': 100.0}, {'country': 'Azerbaijan', 'barrels_from_russia_in_millions': 12.18, 'reliance_percent':

In [32]:
res = ask_sentinel("Which country is most reliable on Russia for their imports for 2027?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = 2027
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
RETURN importer.name AS country,
       ROUND(russian_vol/1000000, 2) AS barrels_from_russia_in_millions,
       ROUND((russian_vol/total_vol) * 100, 2) AS reliance_percent
ORDER BY reliance_percent DESC
LIMIT 5
Full Context:
[]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1641
Completion Tokens: 953
Total Tokens: 2594
Total Cost (if applicable): $0.0
Context Usage: 2.03%
------------------------------------
Based on total trade for 2027, there is no data available to determine which country is most reliable on Russia for their imports.


In [ ]:
res = ask_sentinel("Find instances where an importer is importing the same commodity from two different exporter at a price difference of >30%.")
print(res)

In [33]:
res = ask_sentinel("Who is the largest exporter of oil?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH exporter, sum(f.barrels) AS total_barrels
RETURN exporter.name AS exporter,
       ROUND(total_barrels / 1000000, 2) AS total_volume_in_millions
ORDER BY total_volume_in_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Saudi Arabia', 'total_volume_in_millions': 2145.19}, {'exporter': 'Russian Federation', 'total_volume_in_millions': 1571.25}, {'exporter': 'Canada', 'total_volume_in_millions': 1518.28}, {'exporter': 'USA', 'total_volume_in_millions': 1321.01}, {'exporter': 'United Arab Emirates', 'total_volume_in_millions': 1318.54}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1736
Completion Tokens: 754
Total Tokens: 2490
Total Cost (if applicable): $0.0
C

In [34]:
res = ask_sentinel("If there is a war in the middle east then which countries imports will be affected and by how much?\
Will India be affected?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WHERE exporter.region = 'Middle East'
WITH importer,
     ROUND(sum(f.barrels) / 1000000, 2) AS total_imports_from_middle_east_millions,
     CASE WHEN importer.name = 'India' THEN true ELSE false END AS is_india_affected
RETURN importer.name AS country,
       total_imports_from_middle_east_millions,
       is_india_affected
ORDER BY total_imports_from_middle_east_millions DESC
LIMIT 5
Full Context:
[{'country': 'China', 'total_imports_from_middle_east_millions': 1789.72, 'is_india_affected': False}, {'country': 'Japan', 'total_imports_from_middle_east_millions': 817.61, 'is_india_affected': False}, {'country': 'India', 'total_imports_from_middle_east_millions': 809.27, 'is_india_affected

In [35]:
res = ask_sentinel("Largest importer of crude oil in South East Asia?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country {region: 'South East Asia'})-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
RETURN importer.name AS importer,
       ROUND(sum(f.barrels) / 1000000, 2) AS total_barrels_in_millions
ORDER BY total_barrels_in_millions DESC
LIMIT 5
Full Context:
[{'importer': 'Thailand', 'total_barrels_in_millions': 402.75}, {'importer': 'Singapore', 'total_barrels_in_millions': 321.91}, {'importer': 'Malaysia', 'total_barrels_in_millions': 162.69}, {'importer': 'Indonesia', 'total_barrels_in_millions': 124.99}, {'importer': 'Philippines', 'total_barrels_in_millions': 47.6}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1737
Completion Tokens: 897
Total Tokens: 2634
Total Cost (if applicable): $0.0
Context Usage: 2.06%
------------

In [25]:
res = ask_sentinel("Largest exporter of crude oil in South East Asia?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WHERE exporter.region = 'South East Asia'
WITH exporter, sum(f.barrels) AS total_barrels
RETURN exporter.name AS exporter,
       ROUND(total_barrels / 1000000, 2) AS total_volume_in_millions
ORDER BY total_volume_in_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Malaysia', 'total_volume_in_millions': 595.5}, {'exporter': 'Indonesia', 'total_volume_in_millions': 31.07}, {'exporter': 'Brunei Darussalam', 'total_volume_in_millions': 24.73}, {'exporter': 'Viet Nam', 'total_volume_in_millions': 17.47}, {'exporter': 'Thailand', 'total_volume_in_millions': 6.5}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1737
Completion Tokens: 1003
Total Tokens: 2740
Total Cost (

In [36]:
res = ask_sentinel("Hoe to bake French bread?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
RETURN "Question not related to trade data" AS message LIMIT 5
Full Context:
[{'message': 'Question not related to trade data'}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1634
Completion Tokens: 589
Total Tokens: 2223
Total Cost (if applicable): $0.0
Context Usage: 1.74%
------------------------------------
Based on total trade for [Year]/[Month YYYY]... The question about how to bake French bread is not related to the trade data provided, so I am unable to answer it.


In [26]:
res = ask_sentinel("If there is a conflict between Iran and Israel then which countries imports will be affected?")
print(res)

#The LLM does not inherently know that a conflict between Iran and Israel might lead to a blockade of the Strait of Hormuz,
#  which would affect all Middle Eastern oil (Saudi, UAE, Kuwait, etc.). 
#It only knows what is in the relationship lines.


# 10. GEOPOLITICAL IMPACT: If a question mentions a conflict or 'risk' in the Middle East (e.g., Iran, Israel, Red Sea):
        # a) First, query direct trade for those specific countries.
        # b) Second, automatically query the total imports from the broader region: 
        #    ['Saudi Arabia', 'United Arab Emirates', 'Iraq', 'Kuwait', 'Qatar', 'Oman'].
        # c) Return BOTH sets of data so the user can see the direct and regional exposure.



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WHERE exporter.name IN ['Iran','Israel']
WITH importer, sum(f.barrels) AS affected_barrels
RETURN importer.name AS importer_country,
       ROUND(affected_barrels/1000000, 2) AS total_affected_volume_millions
ORDER BY total_affected_volume_millions DESC
LIMIT 5
Full Context:
[{'importer_country': 'Netherlands', 'total_affected_volume_millions': 3.94}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1668
Completion Tokens: 1121
Total Tokens: 2789
Total Cost (if applicable): $0.0
Context Usage: 2.18%
------------------------------------
Based on total trade for 2023, the imports affected would be those of the Netherlands, with an estimated impact of 3.94 million barrel